In [1]:
# Célula 1: Importações
import sys
import os
import pathlib
import pandas as pd
import numpy as np
import yaml

In [ ]:
# Detecção automática de caminhos
notebook_path = pathlib.Path().absolute()
root_folder = notebook_path.parent

# Definir caminhos
SRC_DIR = root_folder / "src/data_processing"
DATA_DIR = root_folder / "data"
COMPLETE_SERIES_DIR = DATA_DIR / "complete_series"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = root_folder / "config"
FORECAST_DIR = DATA_DIR / "forecast"            # <--- NOVO
#MERGED_DIR = DATA_DIR / "merged_series"         # <--- NOVO
CONFIG_DIR = root_folder / "config"

# Adicionar src ao path
sys.path.append(str(SRC_DIR))

In [ ]:
#from data_processing.import_and_merge_forecast import load_forecast_data, merge_forecast_with_observed
#from data_processing.interpolate_series import load_station_data

In [5]:
# Carregar configurações do arquivo YAML
config_path = CONFIG_DIR / "data_config.yaml"

try:
    # Usar o config_loader para carregar e validar
    sys.path.insert(0, str(root_folder / "src/utils"))
    from config_loader import load_feature_config, load_split_config, ConfigLoader

    sys.path.insert(0, str(root_folder / "src/data_processing"))
    from features_processing import process_features
    
    # Carregar configurações de features
    feature_config = load_feature_config(validate=True)
    
    print("✅ Configurações de features carregadas do arquivo YAML:")
    print(f"Arquivo: {config_path}")
    print()
    
    for key, value in feature_config.items():
        print(f"  - {key}: {value}")
    
    # Extrair configurações de features
    api_k_list = feature_config['api_k_list']
    precipitation_ma_windows = feature_config['precipitation_ma']
    precipitation_cumulative_windows = feature_config['precipitation_cum']
    forecast_ma_windows = feature_config.get('forecast_ma', precipitation_ma_windows)
    forecast_cumulative_windows = feature_config.get('forecast_cum', precipitation_cumulative_windows)
    evapotranspiration_ma_windows = feature_config['evapotranspiration_ma']
    anomaly_ma_windows = feature_config['anomaly_ma']
    
    # Carregar configurações de split
    split_config = load_split_config(validate=True)
    
    print("\n✅ Configurações de split carregadas:")
    for key, value in split_config.items():
        print(f"  - {key}: {value}")
    
except FileNotFoundError as e:
    print(f"❌ ERRO CRÍTICO: {e}")
    print("\nSolução:")
    print("1. Crie o arquivo data_config.yaml na pasta config/")
    print("2. Adicione as seções 'feature_windows' e 'split_config'")
    print("3. Execute novamente o notebook")
    
    # Sugerir criação do arquivo
    print("\nPara criar um arquivo de configuração padrão, execute:")
    print(f"  from config_loader import ConfigLoader")
    print(f"  ConfigLoader.create_default_config(pathlib.Path('config'))")
    
    # Parar a execução
    raise
    
except ValueError as e:
    print(f"❌ ERRO DE CONFIGURAÇÃO: {e}")
    print("\nSolução:")
    print(f"1. Edite o arquivo: {config_path}")
    print("2. Corrija as configurações conforme indicado acima")
    print("3. Execute novamente o notebook")
    raise
    
except ImportError as e:
    print(f"❌ ERRO DE IMPORT: {e}")
    print("\nSolução:")
    print("1. Verifique se o módulo config_loader existe em src/utils/")
    print("2. Verifique se o arquivo data_config.yaml existe em config/")
    raise

✅ Configurações de features carregadas do arquivo YAML:
Arquivo: c:\Users\emily\Documents\GitHub\Mini-curso-GitHub-Leo\config\data_config.yaml

  - api_k_list: [0.7, 0.8, 0.85, 0.9, 0.92, 0.95]
  - precipitation_ma: [3, 7, 15]
  - precipitation_cum: [3, 5, 7, 10]
  - evapotranspiration_ma: [7, 14, 30]
  - anomaly_ma: [3, 7]
  - forecast_ma: [3, 7, 15]
  - forecast_cum: [3, 5, 7, 10]

✅ Configurações de split carregadas:
  - train_ratio: 0.95
  - val_ratio: 0.025
  - test_ratio: 0.025
  - gap: 128
  - window_stride: 1


In [9]:
#Iniciando merge de dados observados com forecast

# Obter lista de estações (assumindo que está no config ou carregue novamente)
# Se não tiver a variável 'stations' disponível, carregue do yaml:
# from src.utils.config_loader import load_config
# data_config = load_config(CONFIG_DIR / "data_config.yaml")
# stations = data_config["stations"]
# Mas geralmente feature_config já tem ou você pode listar os arquivos.
# Vou assumir que você tem a lista 'stations' ou vai carregar aqui.

with open(CONFIG_DIR / "data_config.yaml", 'r') as f:
    d_conf = yaml.safe_load(f)
    stations = d_conf['stations']

for station_id in stations:
    print(f"  Processando estação {station_id}...")
    
    # 2. Carregar forecast
    df_precip, df_et = load_forecast_data(FORECAST_DIR, station_id)

  Processando estação 10100000...
  Processando estação 13150000...
  Processando estação 14100000...


In [19]:
df_precip['date']

0      1995-01-01
1      1995-01-02
2      1995-01-03
3      1995-01-04
4      1995-01-05
          ...    
6343   2012-05-14
6344   2012-05-15
6345   2012-05-16
6346   2012-05-17
6347   2012-05-18
Name: date, Length: 6348, dtype: datetime64[ns]

In [ ]:
    if df_precip is not None and df_et is not None:
        # 3. Merge
        df_merged = merge_forecast_with_observed(df_obs, df_precip, df_et, station_id)
        
        # 4. Salvar no diretório temporário (merged_series)
        output_file = MERGED_DIR / f"{station_id}_complete_date.csv"
        df_merged.to_csv(output_file, index=False)
        print(f"    ✅ Merge salvo em: {output_file.name} (Total linhas: {len(df_merged)})")
    else:
        # Se não tem forecast, copia o original para a pasta merged para não quebrar o fluxo
        print(f"    ⚠️ Sem forecast para estação {station_id}. Copiando original.")
        output_file = MERGED_DIR / f"{station_id}_complete_date.csv"
        df_obs.to_csv(output_file, index=False)

print("✅ Todos os arquivos fundidos e prontos para processamento.")

'''# Executar processamento
process_features(
    input_dir=MERGED_DIR,       # <--- ALTERADO: Usa a pasta com dados fundidos
    output_dir=PROCESSED_DIR,
    config_path=config_path,
    feature_config=feature_config,
    split_config=split_config
)'''

In [17]:
# Configuração das Estações
station_ids = [10100000, 13150000, 14100000]

# Determinar o período dos dados automaticamente
print("\n🔍 Determinando período dos dados...")
try:
    # Carregar um arquivo de exemplo para determinar datas
    sample_file = COMPLETE_SERIES_DIR / f"{station_ids[0]}_complete_date.csv"
    sample_df = pd.read_csv(sample_file)
    sample_df['date'] = pd.to_datetime(sample_df['date'])
    
    start_date = sample_df['date'].min().strftime("%Y-%m-%d")
    end_date = sample_df['date'].max().strftime("%Y-%m-%d")
    
    print(f"  Período encontrado: {start_date} a {end_date}")
    print(f"  Total de dias: {(pd.to_datetime(end_date) - pd.to_datetime(start_date)).days}")
    
except Exception as e:
    print(f"⚠️  Não foi possível determinar o período automaticamente: {e}")
    # Valores padrão como fallback
    start_date = "1990-01-01"
    end_date = "2024-12-31"
    print(f"  Usando período padrão: {start_date} a {end_date}")

# Calcular datas de split automaticamente
loader = ConfigLoader()
split_dates = loader.calculate_split_dates(start_date, end_date)

print("\n📅 Datas de split calculadas automaticamente:")
print(f"  Train: {split_dates['train_start']} → {split_dates['train_end']} ({split_dates['train_days']} dias, {split_dates['train_days']/split_dates['total_days']*100:.1f}%)")
print(f"  Val:   {split_dates['train_end']} → {split_dates['val_end']} ({split_dates['val_days']} dias, {split_dates['val_days']/split_dates['total_days']*100:.1f}%)")
print(f"  Test:  {split_dates['val_end']} → {split_dates['test_end']} ({split_dates['test_days']} dias, {split_dates['test_days']/split_dates['total_days']*100:.1f}%)")

# Definir train_date_cutoff como o final do conjunto de treino
train_date_cutoff = split_dates['train_end']

print("\n📊 Configurações de janelas por tipo de feature:")
print(f"Precipitação (observada):")
print(f"  - Médias móveis: {precipitation_ma_windows}")
print(f"  - Acumulados: {precipitation_cumulative_windows}")
print(f"\nForecast de precipitação:")
print(f"  - Médias móveis: {forecast_ma_windows}")
print(f"  - Acumulados: {forecast_cumulative_windows}")
print(f"\nEvapotranspiração:")
print(f"  - Médias móveis: {evapotranspiration_ma_windows}")
print(f"\nAnomalias:")
print(f"  - Médias móveis: {anomaly_ma_windows}")
print(f"\nAPI:")
print(f"  - Valores k: {api_k_list}")
# Importar módulo de feature engineering
from features_processing import process_features

# Chamar função de processamento
print("\n🚀 Iniciando processamento de features...")
combined_df = process_features(
    input_dir=COMPLETE_SERIES_DIR,
    forecast_dir=FORECAST_DIR,
    output_dir=PROCESSED_DIR,
    station_ids=station_ids,
    # Não precisamos passar parâmetros individuais, serão carregados do config
    api_k_list=None,  # Será carregado do config
    precipitation_ma_windows=None,  # Será carregado do config
    precipitation_cumulative_windows=None,  # Será carregado do config
    forecast_ma_windows=None,  # Será carregado do config
    forecast_cumulative_windows=None,  # Será carregado do config
    evapotranspiration_ma_windows=None,  # Será carregado do config
    anomaly_ma_windows=None,  # Será carregado do config
    train_date_cutoff=train_date_cutoff,
    output_filename="features_combined.csv",
    use_config_file=True  # Habilitar carregamento do arquivo de configuração
)

print("\n✅ Processamento de features concluído!")
print(f"  Arquivo salvo: {PROCESSED_DIR / 'features_combined.csv'}")
print(f"  Train date cutoff usado: {train_date_cutoff}")


🔍 Determinando período dos dados...
  Período encontrado: 1995-01-01 a 2012-04-30
  Total de dias: 6329

📅 Datas de split calculadas automaticamente:
  Train: 1995-01-01 → 2011-06-18 (6012 dias, 95.0%)
  Val:   2011-06-18 → 2011-11-23 (158 dias, 2.5%)
  Test:  2011-11-23 → 2012-04-30 (159 dias, 2.5%)

📊 Configurações de janelas por tipo de feature:
Precipitação (observada):
  - Médias móveis: [3, 7, 15]
  - Acumulados: [3, 5, 7, 10]

Forecast de precipitação:
  - Médias móveis: [3, 7, 15]
  - Acumulados: [3, 5, 7, 10]

Evapotranspiração:
  - Médias móveis: [7, 14, 30]

Anomalias:
  - Médias móveis: [3, 7]

API:
  - Valores k: [0.7, 0.8, 0.85, 0.9, 0.92, 0.95]

🚀 Iniciando processamento de features...
PROCESSAMENTO DE FEATURES

📊 Carregando dados observados de 3 estações...
✓ 3 estações carregadas

🔮 Carregando dados de forecast...
✓ 3 estações com forecast carregadas

🔗 Fazendo merge observado + forecast...


ValueError: You are trying to merge on object and datetime64[ns] columns for key 'date'. If you wish to proceed you should use pd.concat

In [12]:
for i in range(len(combined_df.columns)):
    print(combined_df.columns[i])

Q_10100000
station_id_x
missing_data_x
precipitation_10100000
evapotranspiration_10100000
precipitation_forecast_10100000
evapotranspiration_forecast_10100000
precipitation_ma3_10100000
precipitation_ma7_10100000
precipitation_ma15_10100000
precipitation_cum3_10100000
precipitation_cum5_10100000
precipitation_cum7_10100000
precipitation_cum10_10100000
precipitation_forecast_ma3_10100000
precipitation_forecast_ma7_10100000
precipitation_forecast_ma15_10100000
precipitation_forecast_cum3_10100000
precipitation_forecast_cum5_10100000
precipitation_forecast_cum7_10100000
precipitation_forecast_cum10_10100000
evapotranspiration_ma7_10100000
evapotranspiration_ma14_10100000
evapotranspiration_ma30_10100000
api_k70_10100000
api_forecast_k70_10100000
api_k80_10100000
api_forecast_k80_10100000
api_k85_10100000
api_forecast_k85_10100000
api_k90_10100000
api_forecast_k90_10100000
api_k92_10100000
api_forecast_k92_10100000
api_k95_10100000
api_forecast_k95_10100000
dQ_dt_10100000
dP_dt_10100000
dP